# REST API Integration with MLflow Model

## Topic Goal

This notebook properly implements **REST API integration** for an MLflow model.



This notebook demonstrates:

1. Train a model.
2. Evaluate the model.
3. Infer model signature.
4. Log model with signature and input example in the **same MLflow run**.
5. Create `model_uri` from the same active run.
6. Register the model using the same `model_uri`.
7. Prepare MLflow serving command.
8. Create a valid JSON payload.
9. Test the endpoint using `requests.post()`.
10. Test the endpoint using `curl`.


## 1. Import Libraries

We import MLflow, pandas, scikit-learn, `requests`, and `json`.

`json` is important because the payload file must be saved as a valid JSON dictionary for curl testing.


In [ ]:
# Import os to create folders and manage local paths.
import os

# Import warnings to keep notebook output clean.
import warnings

# Suppress non-critical warnings.
warnings.filterwarnings("ignore")

# Import json to save REST API payload correctly as a dictionary.
import json

# Import MLflow for tracking, model logging, registry, and serving workflow.
import mlflow

# Import MLflow scikit-learn flavor for logging sklearn pipelines.
import mlflow.sklearn

# Import infer_signature to capture model input and output schema.
from mlflow.models.signature import infer_signature

# Import pandas for reading and processing tabular data.
import pandas as pd

# Import numpy for numerical operations.
import numpy as np

# Import requests to call the MLflow REST API endpoint.
import requests

# Import train_test_split to split dataset into train and test sets.
from sklearn.model_selection import train_test_split

# Import ColumnTransformer for preprocessing categorical and numerical columns differently.
from sklearn.compose import ColumnTransformer

# Import OneHotEncoder for categorical columns.
from sklearn.preprocessing import OneHotEncoder

# Import StandardScaler for numerical columns.
from sklearn.preprocessing import StandardScaler

# Import Pipeline to combine preprocessing and model into one deployable object.
from sklearn.pipeline import Pipeline

# Import RandomForestClassifier for churn prediction.
from sklearn.ensemble import RandomForestClassifier

# Import classification metrics.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


## 2. Configure MLflow and Local Folders

This notebook uses local file-based MLflow tracking.

The model serving endpoint will run separately using this command pattern:

```bash
mlflow models serve -m <model_uri> -p 5001 --env-manager local
```


In [ ]:
# Define experiment name.
EXPERIMENT_NAME = "09_rest_api_integration"

# Define run name.
RUN_NAME = "09_rest_api_integration_same_run_usecase"

# Remove inherited MLflow Project experiment variable if present.
os.environ.pop("MLFLOW_EXPERIMENT_ID", None)

# Remove inherited MLflow run variable if present.
os.environ.pop("MLFLOW_RUN_ID", None)

# Create local MLflow tracking folder.
os.makedirs("mlruns", exist_ok=True)

# Create outputs folder for payload and response files.
os.makedirs("outputs", exist_ok=True)

# Create artifacts folder for any supporting files.
os.makedirs("artifacts", exist_ok=True)

# Configure MLflow to use local file-based tracking.
mlflow.set_tracking_uri("file:./mlruns")

# Define dataset path.
DATA_PATH = "datasets/customer_churn_500.csv"

# Set MLflow experiment.
mlflow.set_experiment(EXPERIMENT_NAME)

# Create registered model name.
REGISTERED_MODEL_NAME = EXPERIMENT_NAME.replace("-", "_").replace(" ", "_") + "_Model"

# Define MLflow serving host.
serving_host = "127.0.0.1"

# Define MLflow serving port.
serving_port = 5001

# Define MLflow REST invocation URL.
serving_url = f"http://{serving_host}:{serving_port}/invocations"

# Print setup details.
print("Experiment:", EXPERIMENT_NAME)
print("Run name:", RUN_NAME)
print("Registered model name:", REGISTERED_MODEL_NAME)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Dataset path:", DATA_PATH)
print("Serving URL:", serving_url)


## 3. Load and Prepare Dataset

We use the customer churn dataset.

The model predicts whether a customer is likely to churn.


In [ ]:
# Load customer churn dataset.
df = pd.read_csv(DATA_PATH)

# Display first five records.
display(df.head())

# Define target column.
target_column = "churn"

# Create feature matrix by dropping customer ID and target column.
X = df.drop(columns=["customer_id", target_column])

# Create target vector.
y = df[target_column]

# Identify categorical columns.
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

# Identify numerical columns.
numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

# Print feature groups for explanation.
print("Categorical columns:", categorical_columns)
print("Numerical columns:", numerical_columns)

# Create preprocessing transformer.
preprocessor = ColumnTransformer(
    transformers=[
        # Standardize numerical columns.
        ("num", StandardScaler(), numerical_columns),

        # One-hot encode categorical columns.
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ]
)

# Split data into train and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Print dataset details.
print("Dataset shape:", df.shape)
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


## 4. Train and Evaluate Model

We train a complete scikit-learn pipeline.

The pipeline includes preprocessing and model training so the served model can accept raw input columns.


In [ ]:
# Create Random Forest classifier.
model = RandomForestClassifier(
    n_estimators=150,
    max_depth=6,
    random_state=42
)

# Create complete pipeline with preprocessing and model.
pipeline = Pipeline(steps=[
    # Apply preprocessing to raw input columns.
    ("preprocessor", preprocessor),

    # Train the Random Forest model.
    ("model", model),
])

# Train the pipeline.
pipeline.fit(X_train, y_train)

# Generate predictions on test data.
predictions = pipeline.predict(X_test)

# Calculate accuracy.
accuracy = accuracy_score(y_test, predictions)

# Calculate precision.
precision = precision_score(y_test, predictions, zero_division=0)

# Calculate recall.
recall = recall_score(y_test, predictions, zero_division=0)

# Calculate F1-score.
f1 = f1_score(y_test, predictions, zero_division=0)

# Create classification report.
report = classification_report(y_test, predictions, zero_division=0)

# Print metrics.
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print(report)


## 5. Infer Model Signature

The model signature documents the expected input and output schema.

This is useful for model serving because the API caller must send the correct column names and values.


In [ ]:
# Create input example for signature.
input_example = X_test.head(5)

# Generate sample predictions for the input example.
sample_predictions = pipeline.predict(input_example)

# Infer input and output signature.
signature = infer_signature(input_example, sample_predictions)

# Display input example.
display(input_example)

# Print sample predictions.
print("Sample predictions:", sample_predictions)


## 6. Same-Run Model Logging, Signature, Model URI, and Registry

This section logs everything into the **same MLflow run**:

- parameters
- metrics
- classification report
- model
- input example
- signature
- model URI

Then the model is registered using the same model URI.


In [ ]:
# Start the SAME MLflow run for metrics, model, signature, and model URI.
with mlflow.start_run(run_name=RUN_NAME):

    # Log dataset path.
    mlflow.log_param("dataset", DATA_PATH)

    # Log topic name.
    mlflow.log_param("topic", EXPERIMENT_NAME)

    # Log model family.
    mlflow.log_param("model_family", "RandomForestClassifier")

    # Log serving host.
    mlflow.log_param("serving_host", serving_host)

    # Log serving port.
    mlflow.log_param("serving_port", serving_port)

    # Log serving URL.
    mlflow.log_param("serving_url", serving_url)

    # Log accuracy.
    mlflow.log_metric("accuracy", accuracy)

    # Log precision.
    mlflow.log_metric("precision", precision)

    # Log recall.
    mlflow.log_metric("recall", recall)

    # Log F1-score.
    mlflow.log_metric("f1_score", f1)

    # Save classification report locally.
    with open("outputs/classification_report.txt", "w") as file:
        file.write(report)

    # Log classification report artifact.
    mlflow.log_artifact("outputs/classification_report.txt")

    # Log model WITH input example and signature.
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        input_example=input_example,
        signature=signature
    )

    # Create model URI from the same active run.
    model_uri = f"runs:/{mlflow.active_run().info.run_id}/model"

# Register model using the same-run model URI.
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=REGISTERED_MODEL_NAME
)

# Print same-run model URI.
print("Same-run model URI:", model_uri)

# Print registered model name.
print("Registered model name:", registered_model.name)

# Print registered model version.
print("Registered model version:", registered_model.version)


## 7. Prepare MLflow Serving Command

To test REST API integration, start the model server in a **separate terminal**.

Use the printed command below.

On Mac/Linux terminal, run:

```bash
mlflow models serve -m <model_uri> -p 5001 --env-manager local
```

On Windows, the same command works in Command Prompt or PowerShell.


In [ ]:
# Create the MLflow serving command.
serving_command = f"mlflow models serve -m {model_uri} -p {serving_port} --env-manager local"

# Save serving command to a text file.
with open("outputs/mlflow_serving_command.txt", "w") as file:
    file.write(serving_command)

# Print serving command.
print("Run this command in a separate terminal:")
print(serving_command)

# Print endpoint URL.
print("After server starts, endpoint will be:")
print(serving_url)


## 8. Prepare REST API Payload Correctly

MLflow serving expects a **JSON dictionary** with exactly one of these keys:

```text
instances
inputs
dataframe_split
dataframe_records
```

For pandas-like tabular input, we will use:

```text
dataframe_split
```

Important fix:

Do **not** save the payload using:

```python
pd.Series([payload]).to_json(...)
```

That creates a JSON list and causes this error:

```text
BAD_REQUEST: Received a list.
```

Instead, use:

```python
json.dump(payload, file, indent=4)
```


In [ ]:
# Select sample records for API testing.
sample_input = X_test.head(2)

# Convert sample input into MLflow dataframe_split payload format.
payload = {
    "dataframe_split": sample_input.to_dict(orient="split")
}

# Save payload as a proper JSON dictionary, not as a list.
with open("outputs/sample_payload.json", "w") as file:
    json.dump(payload, file, indent=4)

# Display sample input.
display(sample_input)

# Print payload dictionary.
print(payload)

# Show the exact JSON file content.
with open("outputs/sample_payload.json", "r") as file:
    print(file.read())


## 9. Test with Curl

After starting the MLflow serving server in a separate terminal, run this curl command from the same folder.

For Mac/Linux:

```bash
curl -X POST http://127.0.0.1:5001/invocations -H "Content-Type: application/json" -d @outputs sample_payload.json
```

For Windows Command Prompt:

```bat
curl -X POST http://127.0.0.1:5001/invocations ^
  -H "Content-Type: application/json" ^
  -d @outputs/sample_payload.json
```

If you open `/invocations` in the browser, you will get `Method Not Allowed` because the browser sends GET, but MLflow expects POST.


In [ ]:
# Print Mac/Linux curl command.
print("Mac/Linux curl command:")
print("curl -X POST http://127.0.0.1:5001/invocations \\")
print('  -H "Content-Type: application/json" \\')
print("  -d @outputs/sample_payload.json")

# Print Windows curl command.
print("\nWindows Command Prompt curl command:")
print("curl -X POST http://127.0.0.1:5001/invocations ^")
print('  -H "Content-Type: application/json" ^')
print("  -d @outputs/sample_payload.json")


## 10. REST API Client Call Using Python

This is the actual REST API integration step using Python.

Important:

1. First run the serving command from Section 7 in another terminal.
2. Wait until the server starts.
3. Then run the next cell.

If the server is not running, the cell will show a friendly connection error.


In [ ]:
# Try calling the MLflow model serving endpoint.
try:
    # Send POST request to MLflow serving endpoint.
    response = requests.post(
        serving_url,
        json=payload,
        headers={"Content-Type": "application/json"},
        timeout=10
    )

    # Print HTTP status code.
    print("Status Code:", response.status_code)

    # Try to print JSON response.
    try:
        # Print model prediction response.
        print("Prediction Response:", response.json())
    except Exception:
        # Print raw text if response is not JSON.
        print("Raw Response:", response.text)

    # Save response text to file.
    with open("outputs/rest_api_response.txt", "w") as file:
        file.write(response.text)

except requests.exceptions.ConnectionError:
    # Print friendly message if server is not running.
    print("Connection error: MLflow model server is not running.")
    print("Start the server using this command first:")
    print(serving_command)

except requests.exceptions.Timeout:
    # Print friendly message if request times out.
    print("Request timed out. Check if MLflow model server is running properly.")

except Exception as error:
    # Print any other unexpected error.
    print("Unexpected error while calling REST API:")
    print(error)


## 11. Optional: Log REST API Test Evidence

The model training run is already closed because the serving test happens after starting the server manually.

This optional run stores:

- serving command
- payload file
- response file

This is only API test evidence. The model itself remains logged and registered from the same model run.


In [ ]:
# Start a small optional run only for REST API test evidence.
with mlflow.start_run(run_name="rest_api_test_evidence"):

    # Log model URI being tested.
    mlflow.log_param("tested_model_uri", model_uri)

    # Log serving URL.
    mlflow.log_param("serving_url", serving_url)

    # Log serving command artifact.
    mlflow.log_artifact("outputs/mlflow_serving_command.txt")

    # Log payload file.
    if os.path.exists("outputs/sample_payload.json"):
        mlflow.log_artifact("outputs/sample_payload.json")

    # Log response file if it exists.
    if os.path.exists("outputs/rest_api_response.txt"):
        mlflow.log_artifact("outputs/rest_api_response.txt")

# Print completion.
print("Optional REST API test evidence run completed.")



- REST API integration means external systems call the model over HTTP.
- MLflow serving exposes the model at `/invocations`.
- `/invocations` accepts POST, not GET.
- Opening `/invocations` in a browser shows `Method Not Allowed`.
- The request must be sent as JSON.
- The payload must be a dictionary, not a list.
- `dataframe_split` is a common payload format for pandas-like input.
- Use `json.dump()` to save the payload file correctly.
- The model run contains training metrics, model artifact, input example, signature, and registry source.
- The optional API-test run is only for storing request/response evidence after manual serving.
